### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os 

from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver

from langchain.messages import HumanMessage, SystemMessage

## Message based summarization

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')


agent= create_agent(
    model='gpt-4o-mini',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("messages", 5),
            keep=("messages", 2)
        )
    ]
)

In [3]:
## run with thread id

config={"configurable" : {"thread_id": "test-1"}}


In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:

    response = agent.invoke({"messages": [HumanMessage(content=q)] }, config)

    print(f"Messages:{response}")

    print(f"Messages: {len(response['messages'])}")


Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f522d58e-c8a8-4ace-ab02-c69b85a7d562'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c2f5782bd4', 'id': 'chatcmpl-DkuaWSNh5XIuOEmCDwCBzgGNgf1fy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e749e-284b-7322-9476-55daf25bf098-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

In [7]:
### Based on token size

from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to user more tokens"""

    return f"""Top hotels in {city}:
1. The Grand Plaza - A 5-star luxury hotel in the heart of {city}, featuring rooftop dining, a full-service spa, and panoramic skyline views from every suite.
2. Riverside Boutique Inn - A charming mid-range stay along the {city} waterfront, known for its locally-sourced breakfast, cozy reading lounge, and walkable access to museums and cafes.
3. Skyline Business Suites - A modern hotel near the {city} convention district, offering high-speed internet, dedicated workspaces, and 24/7 concierge service tailored for business travelers."""


agent = create_agent(
    model='gpt-4o-mini',
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)


## run with thread id

config={"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars //4 # 4 chars = 1 token


In [ ]:
## run test

cities = ['paris', 'london', 'Hyderabad']

for city in cities:
    response = agent.invoke(
        {'messages' : [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: _ {tokens} {len(response['messages'])} messages")

    print(f"{(response['messages'])}")

In [10]:
### Fraction

### Based on token size

from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to user more tokens"""

    return f"""Top hotels in {city} The Grand Plaza - A 5-star luxury hotel in the heart of {city}, featuring rooftop dining, a full-service spa, and panoramic skyline views from every suite."""


agent = create_agent(
    model='gpt-4o-mini',
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("fraction", 0.005), # 0.5% = ~ 640 tokens
            keep=("fraction", 0.002), # 0.2% ~ 256 tokens
        )
    ]
)


## run with thread id

config={"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars //4 # 4 chars = 1 token


In [12]:
## run test

cities = ['paris', 'london', 'Hyderabad']

for city in cities:
    response = agent.invoke(
        {'messages' : [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response['messages'])
    fraction = tokens / 128000 # gpt-40-mini context
    print(f"{city}: _ {tokens} {len(response['messages'])} messages")

    print(f"{(response['messages'])}")

paris: _ 477 16 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='c610a54d-d7fd-45d7-8e53-e916631db9d2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_53ee395669', 'id': 'chatcmpl-Dkuv2jtOwM9sxYBfoq8hR7GTdfoe7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e74b1-94c9-7123-b7f2-4a9ec3ded7e2-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_OwnBbokFL51fSZM6f51drfJr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'out